In [26]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2025, 12, 29, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 17, 30))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)

CACHE HIT...: 100%|██████████| 1/1 [00:00<00:00, 51.19it/s]


In [25]:
from SDRUtils.products.usd.filters import SOFR_UNDERLIER_NAMES, SOFR_UNDERLIER_NAMES_MISC, SOFR_FISN_VALUES

def sofr_swap_trades(
    df: pd.DataFrame,
    *,
    include_misc: bool = False,
) -> pd.DataFrame:
    underlier_names = list(SOFR_UNDERLIER_NAMES)
    if include_misc:
        underlier_names.extend(SOFR_UNDERLIER_NAMES_MISC)

    df = df.copy()
    mask = (
        df["UPI Underlier Name"].isin(underlier_names)
        & df["UPI FISN"].isin(SOFR_FISN_VALUES)
        # & df["Action type"].isin(list(action_types))
    )
    return df[mask]

sofr_swap_trades(df)["Action type"].value_counts(), sofr_swap_trades(df)["Event type"].value_counts()

(Action type
 NEWT    1871
 MODI     188
 CORR     112
 TERM      55
 EROR      15
 Name: count, dtype: int64,
 Event type
 TRAD    2039
          127
 EXER      32
 ETRM      22
 NOVA      16
 CLRG       5
 Name: count, dtype: int64)

In [28]:
from SDRUtils.products import USD_SOFR_SwapProduct

product = USD_SOFR_SwapProduct()
cdf = product.build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True)
cdf

FETCHING DELIVERY BASKETS...: 100%|██████████| 6/6 [00:01<00:00,  3.85it/s]


,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,is_mac,is_spreadover,is_asset_swap,risk
0,1570142946000000101,2025-12-29 12:03:13+00:00,2026-08-06,2036-08-06 00:00:00,OIS_SWAP,10.147222,10Y,True,0.611111,7M,...,NaN,NaN,NaN,2036-08-06,NaN,NaN,False,False,False,2500.0
1,1570299802000000201,2025-12-29 12:05:58+00:00,2026-03-18,2046-03-18 00:00:00,OIS_SWAP,20.294444,IMM_H2046,True,0.219444,IMM_H2026,...,NaN,NaN,NaN,2046-03-18,NaN,NaN,False,False,False,1400.0
2,1570469108000000701,2025-12-29 12:09:34+00:00,2026-03-18,2051-03-18 00:00:00,OIS_SWAP,25.369444,IMM_H2051,True,0.219444,IMM_H2026,...,NaN,NaN,NaN,2051-03-18,NaN,NaN,False,False,False,1600.0
3,1570184451000000301,2025-12-29 12:09:46+00:00,2026-03-18,2031-03-18 00:00:00,OIS_SWAP,5.072222,IMM_H2031,True,0.219444,IMM_H2026,...,NaN,NaN,NaN,2031-03-18,NaN,NaN,False,False,False,18200.0
4,1570196242000000201,2025-12-29 12:11:05+00:00,2026-03-18,2036-03-18 00:00:00,OIS_SWAP,10.147222,IMM_H2036,True,0.219444,IMM_H2026,...,NaN,NaN,NaN,2036-03-18,NaN,NaN,False,False,False,9100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1585,1573656731000000101,2025-12-29 21:54:42+00:00,2025-12-31,2026-12-31 00:00:00,OIS_SWAP,1.013889,1Y,False,0.005556,spot,...,91282CME8,2-Year,2024-12-31,2026-12-31,low,NaN,False,False,False,4000.0
1586,1573666981000000101 / 1573660614000000101,2025-12-29T21:55:48+00:00 / 2025-12-29T21:55:3...,2025-12-31,2035-12-31T00:00:00 / 2055-12-31T00:00:00,OIS_SWAP,10.1444 / 30.4361,10Y / 30Y,False,0.005556,spot,...,NaN,NaN,NaN,2035-12-31 / 2055-12-31,NaN,NaN,False,False,False,41900.0
1587,1573691775000000101,2025-12-29 21:59:26+00:00,2025-12-31,2032-12-31 00:00:00,OIS_SWAP,7.102778,7Y,False,0.005556,spot,...,91282CPQ8,7-Year,2025-12-31,2032-12-31,low,NaN,False,False,False,2500.0
1588,1573723141000000201,2025-12-29 21:52:38+00:00,2025-12-31,2055-11-15 00:00:00,OIS_SWAP,30.308333,30Y,False,0.005556,spot,...,912810UP1,30-Year,2025-11-17,2055-11-15,high,NaN,False,False,False,100100.0


In [29]:
# cdf[cdf["trade_id"] == 1573577765000000201].iloc[0].to_dict(), cdf[cdf["trade_id"] == 1573612074000000101].iloc[0].to_dict()
# cdf[(cdf["package_legs"].isna()) & (cdf["Package indicator"] == True) & (cdf["Package transaction spread"].notna()) & (cdf["forward_label"] == "spot")]
cdf[cdf["is_spreadover"] == True].iloc[0:5].to_dict(orient="records")

[{'trade_id': 1570208068000000401,
  'execution_timestamp': Timestamp('2025-12-29 12:12:48+0000', tz='UTC'),
  'effective_date': Timestamp('2025-12-31 00:00:00'),
  'expiration_date': Timestamp('2035-12-31 00:00:00'),
  'product_type': 'OIS_SWAP',
  'tenor_years': 10.144444444444444,
  'tenor_label': '10Y',
  'is_forward': False,
  'forward_start_years': 0.005555555555555556,
  'forward_label': 'spot',
  'trade_label': 'spot 10Y',
  'notional': 5000000.0,
  'notional_currency': 'USD',
  'fixed_rate': 0.0374296,
  'strike': nan,
  'estimated_pv01': 4189.453144700538,
  'package_type': 'OUTRIGHT',
  'package_id': None,
  'upi_underlier_name': 'USD-SOFR-COMPOUND',
  'unique_product_identifier': 'QZXQ4R16245X',
  'platform_identifier': 'BBSF',
  'cleared': 'I',
  'prime_brokerage_transaction_indicator': False,
  'block_trade_election_indicator': False,
  'large_notional_off-facility_swap_election_indicator': None,
  'other_payment_type': '',
  'other_payment_amount': '',
  'package_indicat

In [13]:
import re

col = "UPI Underlier Name"
re.sub(r"(?<!^)(?=[A-Z])", "_", col.lower()).lower().replace(" ", "_")

'upi_underlier_name'